# Exercise 10.3: Benchmark Python vs C++

From *Programming in High Energy Particle Physics*, Chapter 10

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/isildakbora/hep-programming-guide-notebooks/blob/main/ch10/ex10_3_solution.ipynb)

Exercise 10.3 Benchmark Python vs C++ Benchmark Python vs C++ performance by implementing the same event selection in both languages. Time the execution for 1 million events and measure the ratio. Investigate which operations are slower in Python and why.

<details><summary>Hint</summary>

Time the reading and the computation separately ( time.perf_counter() in Python, std::chrono or TStopwatch in C++), and repeat each measurement a few times, since the first run includes file-cache and import effects. Compare at least four Python versions: a per-event Python loop, vectorized Awkward, the loop compiled with @numba.njit , and RDataFrame called from Python. Also try the C++ version with and without ROOT::EnableImplicitMT() .

</details>

**Colab setup:** ROOT installation takes about 3–5 minutes and restarts the runtime once. After the restart, rerun the notebook from the top; the setup cells safely detect an existing mamba/ROOT installation.

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    import shutil
    if not shutil.which("mamba"):
        get_ipython().system("pip install -q condacolab")
        import condacolab
        condacolab.install()  # restarts runtime once

In [ ]:
if IN_COLAB:
    try:
        import ROOT
    except ImportError:
        get_ipython().system("mamba install -q -y -c conda-forge root")
        import ROOT

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    get_ipython().run_line_magic("pip", "install -q uproot awkward numpy numba matplotlib")

**Data:** one million deterministic toy rows. Requires ROOT ≥ 6.30; C++ code is executed by PyROOT.

### Step 1: Generate one million deterministic events

In [ ]:
import ROOT, time, numpy as np, uproot, awkward as ak
N=1_000_000
ROOT.RDataFrame(N).Define('pt','20.0+double((rdfentry_*37)%100)').Define('eta','double(int((rdfentry_*13)%50)-25)/10.0').Define('charge','rdfentry_%2==0 ? 1 : -1').Snapshot('Events','bench.root',['pt','eta','charge'])
t=time.perf_counter(); arr=uproot.open('bench.root')['Events'].arrays(['pt','eta','charge'],library='np'); read_s=time.perf_counter()-t
print('Reading [s]',round(read_s,3))

### Step 2: Time reading and four selection implementations

In [ ]:
from numba import njit
from statistics import median
pt,eta,charge=(arr[k] for k in ['pt','eta','charge'])
def py_loop():
    count=0
    for i in range(N):
        if pt[i]>40 and abs(eta[i])<2.4 and charge[i]>0: count+=1
    return count
def awkward_vec():
    a=ak.Array(arr); return int(ak.sum((a.pt>40)&(abs(a.eta)<2.4)&(a.charge>0)))
@njit
def numba_loop(pt,eta,charge):
    count=0
    for i in range(len(pt)):
        if pt[i]>40 and abs(eta[i])<2.4 and charge[i]>0: count+=1
    return count
numba_loop(pt[:10],eta[:10],charge[:10])
def rdf_python(): return int(ROOT.RDataFrame('Events','bench.root').Filter('pt>40 && abs(eta)<2.4 && charge>0').Count().GetValue())
def measure(fn):
    times=[]; count=0
    for _ in range(3):
        t=time.perf_counter(); count=fn(); times.append(time.perf_counter()-t)
    return count,median(times)
timings={name:measure(fn) for name,fn in [('Python loop',py_loop),('Awkward',awkward_vec),('Numba',lambda:numba_loop(pt,eta,charge)),('PyROOT RDF',rdf_python)]}
assert len({v[0] for v in timings.values()})==1
print(timings)

### Step 3: Run the selection in C++ with and without implicit multithreading

In [ ]:
from pathlib import Path
cpp=r"""#include <ROOT/RDataFrame.hxx>
#include <TROOT.h>
#include <chrono>
#include <iostream>
void bench() {
  for (int n : {0,2,4}) {
    if (n) ROOT::EnableImplicitMT(n); else ROOT::DisableImplicitMT();
    for (int rep=0; rep<4; ++rep) {
      ROOT::RDataFrame df("Events","bench.root");
      auto t=std::chrono::steady_clock::now();
      auto count=*df.Filter("pt>40 && abs(eta)<2.4 && charge>0").Count();
      auto sec=std::chrono::duration<double>(std::chrono::steady_clock::now()-t).count();
      if (rep>0) std::cout << "C++ threads=" << n << " count=" << count << " seconds=" << sec << std::endl;
    }
    ROOT::DisableImplicitMT();
  }
}
"""
Path('bench.C').write_text(cpp); ROOT.gROOT.ProcessLine('.x bench.C')
for name,(count,sec) in timings.items(): print(name,'selection [s]',round(sec,3),'ratio to Awkward',round(sec/timings['Awkward'][1],2))
import matplotlib.pyplot as plt
fig,ax=plt.subplots(figsize=(7,4))
names=list(timings); ax.bar(names,[timings[k][1] for k in names]); ax.set(ylabel='Median selection time [s]',xlabel='Implementation')
ax.tick_params(axis='x',rotation=25); fig.tight_layout(); plt.show()

Reading is reported separately. Selection counts are checked before comparing times. The first RDataFrame call can include JIT and file-cache effects; repeat the benchmark under fixed hardware for a reliable ratio.